#Log rank Implementation

##GPT2-XL Backbone

In [ ]:
import os, sys, torch, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from datasets import load_dataset
import ast

# MODEL SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"
BACKBONE = "gpt2-xl"
print(f"Using {BACKBONE} on {device}")

model = AutoModelForCausalLM.from_pretrained(
    BACKBONE,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
).eval()

tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# LOG-RANK SCORE
def compute_log_rank_score(text, model, tokenizer, device, max_len=512):

    try:
        inp = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_len,
            padding=False
        ).to(device)

        ids = inp["input_ids"]
        if ids.shape[1] < 2:
            return np.nan

        with torch.no_grad():
            logits = model(ids).logits

        log_ranks = []
        for i in range(ids.shape[1] - 1):
            pos_logits = logits[0, i, :]
            target_token = ids[0, i + 1].item()
            rank = (pos_logits > pos_logits[target_token]).sum().item() + 1
            log_ranks.append(np.log(rank))

        return np.mean(log_ranks) if log_ranks else np.nan
    except:
        return np.nan

# PREPARE FUNCTION (handles LLM-edited columns)
def prepare(df, col1, col2):

    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)
            if pd.isna(val):
                continue
            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                                    break
                except:
                    text = str(val).strip()
                    if text:
                        X.append(text)
                        y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y

# EVALUATION HELPER

def evaluate_logrank(texts, labels, model, tokenizer, device):
    scores = []
    for t in tqdm(texts, desc="Scoring", ncols=80):
        s = compute_log_rank_score(t, model, tokenizer, device)
        scores.append(5.0 if (np.isnan(s) or np.isinf(s)) else s)
    scores = [-s for s in scores]
    thr = np.median([s for s in scores if np.isfinite(s)])
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    auroc = roc_auc_score(labels, scores)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )

    return scores, preds, {
        'accuracy': acc,
        'auroc': auroc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'threshold': thr
    }

# LOAD BEEMO DATASET
df = load_dataset("toloka/beemo")['train'].to_pandas()

tasks = {
    'H_vs_M': ('human_output', 'model_output'),
    'E_vs_M': ('human_edits', 'model_output'),
    'H_vs_E': ('human_output', 'human_edits'),
    'L_vs_M': ('llama-3.1-70b_edits', 'model_output'),
    'H_vs_L': ('human_output', 'llama-3.1-70b_edits'),
    'G_vs_M': ('gpt-4o_edits', 'model_output'),
    'H_vs_G': ('human_output', 'gpt-4o_edits'),
}


# RUN ALL TASKS
PROJECT = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT}/results", exist_ok=True)
res = {}

for name, (c1, c2) in tasks.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    X, y = prepare(df, c1, c2)
    print(f"Samples: {len(X)} ({y.count(0)}/{y.count(1)})")

    sc, pr, mt = evaluate_logrank(X, y, model, tokenizer, device)
    res[name] = mt

    bn = BACKBONE.replace("/", "_")
    pd.DataFrame({
        'text': X,
        'label': y,
        'score': sc,
        'pred': pr
    }).to_csv(
        f"{PROJECT}/results/{name}_logrank_{bn}.csv",
        index=False
    )
    print(f"AUROC: {mt['auroc']:.4f} | Acc: {mt['accuracy']:.4f} | F1: {mt['f1']:.4f}")

# SUMMARY
bn = BACKBONE.replace("/", "_")
summary = pd.DataFrame(res).T
summary.to_csv(f"{PROJECT}/results/logrank_{bn}_summary.csv")

print("\n" + "="*60)
print(" FINAL SUMMARY (GPT2XL)")
print("="*60)
print(summary[['auroc', 'accuracy', 'f1']].round(4))
print(f"\n Results saved to {PROJECT}/results/")

Using gpt2-xl on cuda


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

AUROC: 0.5749 | Acc: 0.5401 | F1: 0.6321

 G_vs_M
Samples: 8748 (6561/2187)


Scoring: 100%|██████████████████████████████| 8748/8748 [07:56<00:00, 18.37it/s]


AUROC: 0.6105 | Acc: 0.5497 | F1: 0.3996

 H_vs_G
Samples: 8748 (2187/6561)


Scoring: 100%|██████████████████████████████| 8748/8748 [07:21<00:00, 19.83it/s]


AUROC: 0.6074 | Acc: 0.5639 | F1: 0.6511

 FINAL SUMMARY (GPT2XL)
         auroc  accuracy      f1
H_vs_M  0.7017    0.6443  0.6443
E_vs_M  0.6446    0.5958  0.5958
H_vs_E  0.5725    0.5546  0.5546
L_vs_M  0.6511    0.5714  0.4286
H_vs_L  0.5749    0.5401  0.6321
G_vs_M  0.6105    0.5497  0.3996
H_vs_G  0.6074    0.5639  0.6511

 Results saved to /content/drive/MyDrive/BEEMO_Reproducibility/results/


##OPT-1.3B Backbone

In [ ]:
import os, sys, torch, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from datasets import load_dataset
import ast

# MODEL SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"
BACKBONE = "facebook/opt-1.3b"
if device == "cpu":
    BACKBONE = "gpt2-xl"
print(f"Using {BACKBONE} on {device}")

model = AutoModelForCausalLM.from_pretrained(
    BACKBONE,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
).eval()

tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LOG-RANK SCORE
def compute_log_rank_score(text, model, tokenizer, device, max_len=512):

    try:
        inp = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_len,
            padding=False
        ).to(device)

        ids = inp["input_ids"]
        if ids.shape[1] < 2:
            return np.nan

        with torch.no_grad():
            logits = model(ids).logits

        log_ranks = []
        for i in range(ids.shape[1] - 1):
            pos_logits = logits[0, i, :]
            target_token = ids[0, i + 1].item()
            rank = (pos_logits > pos_logits[target_token]).sum().item() + 1
            log_ranks.append(np.log(rank))

        return np.mean(log_ranks) if log_ranks else np.nan
    except:
        return np.nan

# FIXED PREPARE FUNCTION (handles LLM-edited columns)

def prepare(df, col1, col2):
    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue
            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                                    break
                except:
                    text = str(val).strip()
                    if text:
                        X.append(text)
                        y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y

# EVALUATION HELPER

def evaluate_logrank(texts, labels, model, tokenizer, device):
    scores = []
    for t in tqdm(texts, desc="Scoring", ncols=80):
        s = compute_log_rank_score(t, model, tokenizer, device)
        scores.append(5.0 if (np.isnan(s) or np.isinf(s)) else s)
    scores = [-s for s in scores]

    thr = np.median([s for s in scores if np.isfinite(s)])
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    auroc = roc_auc_score(labels, scores)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )

    return scores, preds, {
        'accuracy': acc,
        'auroc': auroc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'threshold': thr
    }

# LOAD BEEMO DATASET
df = load_dataset("toloka/beemo")['train'].to_pandas()

tasks = {
    'H_vs_M': ('human_output', 'model_output'),
    'E_vs_M': ('human_edits', 'model_output'),
    'H_vs_E': ('human_output', 'human_edits'),
    'L_vs_M': ('llama-3.1-70b_edits', 'model_output'),
    'H_vs_L': ('human_output', 'llama-3.1-70b_edits'),
    'G_vs_M': ('gpt-4o_edits', 'model_output'),
    'H_vs_G': ('human_output', 'gpt-4o_edits'),
}

# RUN ALL TASKS
PROJECT = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT}/results", exist_ok=True)
res = {}

for name, (c1, c2) in tasks.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    X, y = prepare(df, c1, c2)
    print(f"Samples: {len(X)} ({y.count(0)}/{y.count(1)})")

    sc, pr, mt = evaluate_logrank(X, y, model, tokenizer, device)
    res[name] = mt

    bn = BACKBONE.replace("/", "_")
    pd.DataFrame({
        'text': X,
        'label': y,
        'score': sc,
        'pred': pr
    }).to_csv(
        f"{PROJECT}/results/{name}_logrank_{bn}.csv",
        index=False
    )
    print(f"AUROC: {mt['auroc']:.4f} | Acc: {mt['accuracy']:.4f} | F1: {mt['f1']:.4f}")

# SUMMARY
bn = BACKBONE.replace("/", "_")
summary = pd.DataFrame(res).T
summary.to_csv(f"{PROJECT}/results/logrank_{bn}_summary.csv")

print("\n" + "="*60)
print(" FINAL SUMMARY (OPT-1.3B)")
print("="*60)
print(summary[['auroc', 'accuracy', 'f1']].round(4))
print(f"\n Results saved to {PROJECT}/results/")

Using facebook/opt-1.3b on cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!



 H_vs_M
Samples: 4374 (2187/2187)


Scoring: 100%|██████████████████████████████| 4374/4374 [03:31<00:00, 20.69it/s]


AUROC: 0.7160 | Acc: 0.6566 | F1: 0.6566

 E_vs_M
Samples: 4374 (2187/2187)


...

AUROC: 0.5735 | Acc: 0.5358 | F1: 0.6286

 G_vs_M
Samples: 8748 (6561/2187)


Scoring: 100%|██████████████████████████████| 8748/8748 [05:37<00:00, 25.94it/s]


AUROC: 0.6165 | Acc: 0.5548 | F1: 0.4063

 H_vs_G
Samples: 8748 (2187/6561)


Scoring: 100%|██████████████████████████████| 8748/8748 [05:11<00:00, 28.08it/s]


AUROC: 0.6221 | Acc: 0.5676 | F1: 0.6540

 FINAL SUMMARY (OPT-1.3B)
         auroc  accuracy      f1
H_vs_M  0.7160    0.6566  0.6566
E_vs_M  0.6577    0.6109  0.6109
H_vs_E  0.5748    0.5528  0.5528
L_vs_M  0.6705    0.5826  0.4435
H_vs_L  0.5735    0.5358  0.6286
G_vs_M  0.6165    0.5548  0.4063
H_vs_G  0.6221    0.5676  0.6540

 Results saved to /content/drive/MyDrive/BEEMO_Reproducibility/results/


##Falcon-7B Backbone

In [ ]:
import os, sys, torch, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from datasets import load_dataset
import ast

# MODEL SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"
BACKBONE = "tiiuae/falcon-7b"
print(f"Using {BACKBONE} on {device}")

model = AutoModelForCausalLM.from_pretrained(
    BACKBONE,
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
).eval()

tokenizer = AutoTokenizer.from_pretrained(BACKBONE,trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LOG-RANK SCORE
def compute_log_rank_score(text, model, tokenizer, device, max_len=512):

    try:
        inp = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_len,
            padding=False
        ).to(device)

        ids = inp["input_ids"]
        if ids.shape[1] < 2:
            return np.nan

        with torch.no_grad():
            logits = model(ids).logits

        log_ranks = []
        for i in range(ids.shape[1] - 1):
            pos_logits = logits[0, i, :]
            target_token = ids[0, i + 1].item()
            rank = (pos_logits > pos_logits[target_token]).sum().item() + 1
            log_ranks.append(np.log(rank))

        return np.mean(log_ranks) if log_ranks else np.nan
    except:
        return np.nan

# FIXED PREPARE FUNCTION (handles LLM-edited columns)
def prepare(df, col1, col2):
    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue
            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                                    break
                except:
                    text = str(val).strip()
                    if text:
                        X.append(text)
                        y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y

# EVALUATION HELPER
def evaluate_logrank(texts, labels, model, tokenizer, device):
    scores = []
    for t in tqdm(texts, desc="Scoring", ncols=80):
        s = compute_log_rank_score(t, model, tokenizer, device)
        scores.append(5.0 if (np.isnan(s) or np.isinf(s)) else s)
    scores = [-s for s in scores]

    thr = np.median([s for s in scores if np.isfinite(s)])
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    auroc = roc_auc_score(labels, scores)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )

    return scores, preds, {
        'accuracy': acc,
        'auroc': auroc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'threshold': thr
    }

# LOAD BEEMO DATASET
df = load_dataset("toloka/beemo")['train'].to_pandas()

tasks = {
    'H_vs_M': ('human_output', 'model_output'),
    'E_vs_M': ('human_edits', 'model_output'),
    'H_vs_E': ('human_output', 'human_edits'),
    'L_vs_M': ('llama-3.1-70b_edits', 'model_output'),
    'H_vs_L': ('human_output', 'llama-3.1-70b_edits'),
    'G_vs_M': ('gpt-4o_edits', 'model_output'),
    'H_vs_G': ('human_output', 'gpt-4o_edits'),
}

# RUN ALL TASKS
PROJECT = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT}/results", exist_ok=True)
res = {}

for name, (c1, c2) in tasks.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    X, y = prepare(df, c1, c2)
    print(f"Samples: {len(X)} ({y.count(0)}/{y.count(1)})")

    sc, pr, mt = evaluate_logrank(X, y, model, tokenizer, device)
    res[name] = mt

    bn = BACKBONE.replace("/", "_")
    pd.DataFrame({
        'text': X,
        'label': y,
        'score': sc,
        'pred': pr
    }).to_csv(
        f"{PROJECT}/results/{name}_logrank_{bn}.csv",
        index=False
    )
    print(f"AUROC: {mt['auroc']:.4f} | Acc: {mt['accuracy']:.4f} | F1: {mt['f1']:.4f}")

# SUMMARY

bn = BACKBONE.replace("/", "_")
summary = pd.DataFrame(res).T
summary.to_csv(f"{PROJECT}/results/logrank_{bn}_summary.csv")

print("\n" + "="*60)
print(" FINAL SUMMARY (Falcon7b)")
print("="*60)
print(summary[['auroc', 'accuracy', 'f1']].round(4))
print(f"\n Results saved to {PROJECT}/results/")

Using tiiuae/falcon-7b on cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_falcon.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/tiiuae/falcon-7b:
- configuration_falcon.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!



...

AUROC: 0.5596 | Acc: 0.5287 | F1: 0.6230

 G_vs_M
Samples: 8748 (6561/2187)


Scoring: 100%|██████████████████████████████| 8748/8748 [08:51<00:00, 16.47it/s]


AUROC: 0.6009 | Acc: 0.5477 | F1: 0.3969

 H_vs_G
Samples: 8748 (2187/6561)


Scoring: 100%|██████████████████████████████| 8748/8748 [08:23<00:00, 17.37it/s]


AUROC: 0.6119 | Acc: 0.5602 | F1: 0.6482

 FINAL SUMMARY (Falcon7b)
         auroc  accuracy      f1
H_vs_M  0.6955    0.6406  0.6406
E_vs_M  0.6469    0.6049  0.6049
H_vs_E  0.5645    0.5455  0.5455
L_vs_M  0.6597    0.5785  0.4380
H_vs_L  0.5596    0.5287  0.6230
G_vs_M  0.6009    0.5477  0.3969
H_vs_G  0.6119    0.5602  0.6482

 Results saved to /content/drive/MyDrive/BEEMO_Reproducibility/results/
